## 1. Import Libraries and Setup

In [1]:
import json
import numpy as np
import requests
from tqdm.auto import tqdm
from datetime import datetime
import time
import os
from pathlib import Path

print("✓ Libraries imported successfully")

✓ Libraries imported successfully


## 2. Configuration

In [2]:
# API Configuration
QWEN_API_URL = "http://127.0.0.1:8000/embed"
QWEN_INFO_URL = "http://127.0.0.1:8000/"

# File paths
INPUT_FILE = "cleaned_dataset.json"
OUTPUT_EMBEDDINGS = "qwen_embeddings.npy"
OUTPUT_METADATA = "qwen_embeddings_metadata.json"
CHECKPOINT_DIR = "qwen_checkpoints"

# Processing parameters
BATCH_SIZE = 8  # Number of texts to embed per API call
MAX_RETRIES = 3  # Number of retries for failed API calls
RETRY_DELAY = 2  # Seconds to wait before retrying
CHECKPOINT_INTERVAL = 500  # Save checkpoint every N papers

# Embedding parameters
INSTRUCTION = "Scientific paper title and abstract: "  # Task instruction
NORMALIZE = True  # Normalize embeddings to unit vectors

# Create checkpoint directory
Path(CHECKPOINT_DIR).mkdir(exist_ok=True)

print("✓ Configuration set")
print(f"  - API URL: {QWEN_API_URL}")
print(f"  - Batch size: {BATCH_SIZE}")
print(f"  - Checkpoint interval: {CHECKPOINT_INTERVAL}")

✓ Configuration set
  - API URL: http://127.0.0.1:8000/embed
  - Batch size: 8
  - Checkpoint interval: 500


## 3. Test API Connection

In [3]:
# Test connection to Qwen API
try:
    response = requests.get(QWEN_INFO_URL, timeout=5)
    response.raise_for_status()
    info = response.json()
    
    print("✓ Successfully connected to Qwen API")
    print(f"  - Service: {info.get('service', 'N/A')}")
    print(f"  - Embedding model: {info.get('embedding_model', 'N/A')}")
    print(f"  - Device: {info.get('device', 'N/A')}")
    print(f"  - Embedding dimension: {info.get('embedding_dim', 'N/A')}")
    
    EMBEDDING_DIM = info.get('embedding_dim', 1024)
    
except requests.exceptions.RequestException as e:
    print(f"✗ Failed to connect to Qwen API: {e}")
    print("  Please ensure the Qwen API server is running on http://0.0.0.0:8000")
    raise

✓ Successfully connected to Qwen API
  - Service: qwen3-embedding-reranker-api
  - Embedding model: Qwen/Qwen3-Embedding-0.6B
  - Device: cuda
  - Embedding dimension: 1024


## 4. Load Dataset

In [4]:
# Load the cleaned dataset
print(f"Loading dataset from {INPUT_FILE}...")
start_time = time.time()

with open(INPUT_FILE, 'r', encoding='utf-8') as f:
    dataset = json.load(f)

load_time = time.time() - start_time

print(f"✓ Dataset loaded in {load_time:.2f} seconds")
print(f"  - Total papers: {len(dataset):,}")
print(f"  - File size: {os.path.getsize(INPUT_FILE) / (1024**2):.2f} MB")

# Check data structure
if dataset:
    sample = dataset[0]
    print(f"\nSample paper keys: {list(sample.keys())}")
    
    if 'full_text' in sample:
        print(f"✓ 'full_text' field found")
        print(f"  - Sample text length: {len(sample['full_text'])} characters")
        print(f"  - Sample text preview: {sample['full_text'][:200]}...")
    else:
        print("✗ 'full_text' field not found in dataset!")
        raise ValueError("Dataset must contain 'full_text' field")

Loading dataset from cleaned_dataset.json...
✓ Dataset loaded in 1.31 seconds
  - Total papers: 67,043
  - File size: 332.53 MB

Sample paper keys: ['id', 'title', 'abstract', 'authors', 'journal', 'publication_year', 'publication_month', 'publication_day', 'doi', 'keywords', 'mesh', 'language_list', 'embedding', 'full_text', 'cleaned_text']
✓ 'full_text' field found
  - Sample text length: 1538 characters
  - Sample text preview: Immunomodulation by cytokine antisense oligonucleotides. The cytokine network is involved in normal immune reaction and in the progression of several pathologies. Antisense (AS) oligonucleotides, whic...
✓ Dataset loaded in 1.31 seconds
  - Total papers: 67,043
  - File size: 332.53 MB

Sample paper keys: ['id', 'title', 'abstract', 'authors', 'journal', 'publication_year', 'publication_month', 'publication_day', 'doi', 'keywords', 'mesh', 'language_list', 'embedding', 'full_text', 'cleaned_text']
✓ 'full_text' field found
  - Sample text length: 1538 charact

## 5. Define Embedding Functions

In [5]:
def embed_batch(texts, instruction=None, normalize=True, max_retries=MAX_RETRIES):
    """
    Embed a batch of texts using the Qwen API.
    
    Args:
        texts: List of text strings to embed
        instruction: Optional task instruction
        normalize: Whether to normalize embeddings
        max_retries: Maximum number of retry attempts
        
    Returns:
        numpy array of embeddings, shape (len(texts), embedding_dim)
    """
    payload = {
        "texts": texts,
        "instruction": instruction,
        "normalize": normalize
    }
    
    for attempt in range(max_retries):
        try:
            response = requests.post(QWEN_API_URL, json=payload, timeout=120)
            response.raise_for_status()
            
            result = response.json()
            embeddings = np.array(result['embeddings'], dtype=np.float32)
            
            return embeddings
            
        except requests.exceptions.RequestException as e:
            if attempt < max_retries - 1:
                print(f"  ⚠ Attempt {attempt + 1} failed: {e}. Retrying in {RETRY_DELAY}s...")
                time.sleep(RETRY_DELAY)
            else:
                print(f"  ✗ All {max_retries} attempts failed for batch")
                raise
    
    return None


def save_checkpoint(embeddings_list, current_idx, checkpoint_dir=CHECKPOINT_DIR):
    """
    Save a checkpoint of the current progress.
    
    Args:
        embeddings_list: List of embedding arrays
        current_idx: Current index in the dataset
        checkpoint_dir: Directory to save checkpoints
    """
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    checkpoint_file = os.path.join(checkpoint_dir, f"checkpoint_{current_idx}_{timestamp}.npy")
    
    embeddings_array = np.vstack(embeddings_list)
    np.save(checkpoint_file, embeddings_array)
    
    print(f"  💾 Checkpoint saved: {checkpoint_file} (shape: {embeddings_array.shape})")
    
    return checkpoint_file


def load_latest_checkpoint(checkpoint_dir=CHECKPOINT_DIR):
    """
    Load the latest checkpoint if it exists.
    
    Returns:
        (embeddings_array, start_idx) or (None, 0) if no checkpoint found
    """
    checkpoint_files = list(Path(checkpoint_dir).glob("checkpoint_*.npy"))
    
    if not checkpoint_files:
        return None, 0
    
    # Get the latest checkpoint
    latest_checkpoint = max(checkpoint_files, key=lambda p: p.stat().st_mtime)
    
    # Extract index from filename
    filename = latest_checkpoint.stem
    start_idx = int(filename.split('_')[1])
    
    # Load embeddings
    embeddings = np.load(latest_checkpoint)
    
    print(f"✓ Loaded checkpoint: {latest_checkpoint.name}")
    print(f"  - Resume from index: {start_idx}")
    print(f"  - Embeddings shape: {embeddings.shape}")
    
    return embeddings, start_idx


print("✓ Embedding functions defined")

✓ Embedding functions defined


## 6. Check for Existing Checkpoints (Optional Resume)

In [8]:
# Check if we should resume from a checkpoint
RESUME_FROM_CHECKPOINT = True  # Set to False to start fresh

if RESUME_FROM_CHECKPOINT:
    existing_embeddings, start_idx = load_latest_checkpoint()
    if existing_embeddings is not None:
        print(f"\n⚠ Found existing checkpoint. Starting from paper {start_idx}")
        print(f"  Change RESUME_FROM_CHECKPOINT to False to start fresh")
    else:
        print("No existing checkpoints found. Starting from beginning.")
        start_idx = 0
else:
    print("Starting fresh (ignoring any existing checkpoints)")
    existing_embeddings = None
    start_idx = 0

✓ Loaded checkpoint: checkpoint_23736_20251202_193641.npy
  - Resume from index: 23736
  - Embeddings shape: (20704, 1024)

⚠ Found existing checkpoint. Starting from paper 23736
  Change RESUME_FROM_CHECKPOINT to False to start fresh


## 7. Generate Embeddings (Main Processing)

In [9]:
# Extract full texts from dataset
print(f"\nExtracting full texts from dataset...")
full_texts = [paper.get('full_text', '') for paper in dataset]

# Count empty texts
empty_count = sum(1 for text in full_texts if not text.strip())
if empty_count > 0:
    print(f"⚠ Warning: Found {empty_count} papers with empty full_text")

# Initialize embeddings list
if existing_embeddings is not None:
    embeddings_list = [existing_embeddings]
else:
    embeddings_list = []

# Calculate total batches
remaining_texts = full_texts[start_idx:]
total_batches = (len(remaining_texts) + BATCH_SIZE - 1) // BATCH_SIZE

print(f"\n{'='*60}")
print(f"Starting embedding generation")
print(f"{'='*60}")
print(f"Total papers: {len(dataset):,}")
print(f"Starting from: {start_idx:,}")
print(f"Remaining: {len(remaining_texts):,}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Total batches: {total_batches:,}")
print(f"{'='*60}\n")

# Process in batches with progress bar
start_time = time.time()
failed_batches = []

try:
    with tqdm(total=len(remaining_texts), desc="Embedding papers", unit="paper") as pbar:
        for i in range(0, len(remaining_texts), BATCH_SIZE):
            batch_texts = remaining_texts[i:i+BATCH_SIZE]
            actual_idx = start_idx + i
            
            try:
                # Generate embeddings for batch
                batch_embeddings = embed_batch(
                    texts=batch_texts,
                    instruction=INSTRUCTION,
                    normalize=NORMALIZE
                )
                
                embeddings_list.append(batch_embeddings)
                
                # Update progress bar
                pbar.update(len(batch_texts))
                
                # Save checkpoint at intervals
                if (actual_idx + len(batch_texts)) % CHECKPOINT_INTERVAL < BATCH_SIZE:
                    save_checkpoint(embeddings_list, actual_idx + len(batch_texts))
                
            except Exception as e:
                print(f"\n✗ Failed to process batch at index {actual_idx}: {e}")
                failed_batches.append((actual_idx, actual_idx + len(batch_texts)))
                pbar.update(len(batch_texts))
                
except KeyboardInterrupt:
    print("\n\n⚠ Process interrupted by user")
    print("Saving current progress...")
    if embeddings_list:
        save_checkpoint(embeddings_list, start_idx + i)
    raise

# Calculate statistics
elapsed_time = time.time() - start_time
papers_processed = len(remaining_texts) - len(failed_batches) * BATCH_SIZE

print(f"\n{'='*60}")
print(f"Processing complete!")
print(f"{'='*60}")
print(f"Time elapsed: {elapsed_time:.2f}s ({elapsed_time/60:.2f} minutes)")
print(f"Papers processed: {papers_processed:,}")
print(f"Processing rate: {papers_processed/elapsed_time:.2f} papers/second")

if failed_batches:
    print(f"\n⚠ Failed batches: {len(failed_batches)}")
    print(f"  Failed ranges: {failed_batches}")
else:
    print(f"✓ All batches processed successfully!")


Extracting full texts from dataset...

Starting embedding generation
Total papers: 67,043
Starting from: 23,736
Remaining: 43,307
Batch size: 8
Total batches: 5,414



Embedding papers:   0%|          | 0/43307 [00:00<?, ?paper/s]

  💾 Checkpoint saved: qwen_checkpoints\checkpoint_24000_20251202_193927.npy (shape: (20968, 1024))
  💾 Checkpoint saved: qwen_checkpoints\checkpoint_24504_20251202_194039.npy (shape: (21472, 1024))
  💾 Checkpoint saved: qwen_checkpoints\checkpoint_24504_20251202_194039.npy (shape: (21472, 1024))
  💾 Checkpoint saved: qwen_checkpoints\checkpoint_25000_20251202_194202.npy (shape: (21968, 1024))
  💾 Checkpoint saved: qwen_checkpoints\checkpoint_25000_20251202_194202.npy (shape: (21968, 1024))
  💾 Checkpoint saved: qwen_checkpoints\checkpoint_25504_20251202_194310.npy (shape: (22472, 1024))
  💾 Checkpoint saved: qwen_checkpoints\checkpoint_25504_20251202_194310.npy (shape: (22472, 1024))
  💾 Checkpoint saved: qwen_checkpoints\checkpoint_26000_20251202_194427.npy (shape: (22968, 1024))
  💾 Checkpoint saved: qwen_checkpoints\checkpoint_26000_20251202_194427.npy (shape: (22968, 1024))
  💾 Checkpoint saved: qwen_checkpoints\checkpoint_26504_20251202_194611.npy (shape: (23472, 1024))
  💾 Checkp

In [10]:
# Safety Check: Verify all embeddings exist and are valid
print("Running safety checks on embeddings...\n")

# Check 1: Verify we have embeddings for all papers
expected_count = len(full_texts) if start_idx == 0 else len(full_texts)
actual_count = sum(emb.shape[0] for emb in embeddings_list)

print(f"Check 1: Count verification")
print(f"  - Expected embeddings: {expected_count:,}")
print(f"  - Actual embeddings: {actual_count:,}")

if actual_count == expected_count:
    print(f"  ✓ All papers have embeddings\n")
elif actual_count < expected_count:
    missing = expected_count - actual_count
    print(f"  ⚠ Missing {missing:,} embeddings!\n")
else:
    print(f"  ⚠ More embeddings than papers ({actual_count - expected_count:,} extra)!\n")

# Check 2: Verify embedding dimensions are consistent
print(f"Check 2: Dimension verification")
dimensions = [emb.shape[1] for emb in embeddings_list]
unique_dims = set(dimensions)

if len(unique_dims) == 1:
    print(f"  ✓ All embeddings have dimension: {dimensions[0]}")
    if dimensions[0] == EMBEDDING_DIM:
        print(f"  ✓ Matches expected dimension: {EMBEDDING_DIM}\n")
    else:
        print(f"  ⚠ Expected dimension {EMBEDDING_DIM}, got {dimensions[0]}\n")
else:
    print(f"  ✗ Inconsistent dimensions found: {unique_dims}\n")

# Check 3: Check for NaN or Inf values
print(f"Check 3: Validity check (NaN/Inf)")
has_issues = False
for idx, emb_batch in enumerate(embeddings_list):
    nan_count = np.isnan(emb_batch).sum()
    inf_count = np.isinf(emb_batch).sum()
    
    if nan_count > 0 or inf_count > 0:
        print(f"  ✗ Batch {idx}: {nan_count} NaN values, {inf_count} Inf values")
        has_issues = True

if not has_issues:
    print(f"  ✓ No NaN or Inf values found\n")
else:
    print(f"  ⚠ Invalid values detected!\n")

# Check 4: Verify normalization (if enabled)
if NORMALIZE:
    print(f"Check 4: Normalization verification")
    all_norms_valid = True
    
    for idx, emb_batch in enumerate(embeddings_list):
        norms = np.linalg.norm(emb_batch, axis=1)
        # Check if norms are close to 1.0 (within tolerance)
        if not np.allclose(norms, 1.0, atol=1e-5):
            print(f"  ⚠ Batch {idx}: Norms not normalized (mean: {norms.mean():.6f})")
            all_norms_valid = False
    
    if all_norms_valid:
        print(f"  ✓ All embeddings are properly normalized\n")
    else:
        print(f"  ⚠ Some embeddings are not normalized!\n")

# Check 5: Verify no zero vectors
print(f"Check 5: Zero vector check")
zero_vectors = 0
for emb_batch in embeddings_list:
    # Check if any row is all zeros
    zero_mask = ~emb_batch.any(axis=1)
    zero_vectors += zero_mask.sum()

if zero_vectors == 0:
    print(f"  ✓ No zero vectors found\n")
else:
    print(f"  ⚠ Found {zero_vectors} zero vectors!\n")

# Check 6: Sample embedding range check
print(f"Check 6: Value range check")
all_values = np.concatenate([emb.flatten() for emb in embeddings_list])
print(f"  - Min value: {all_values.min():.6f}")
print(f"  - Max value: {all_values.max():.6f}")
print(f"  - Mean value: {all_values.mean():.6f}")
print(f"  - Std deviation: {all_values.std():.6f}")

if NORMALIZE:
    # For normalized vectors, values should typically be in [-1, 1]
    if all_values.min() >= -1.0 and all_values.max() <= 1.0:
        print(f"  ✓ Values within expected range for normalized embeddings [-1, 1]\n")
    else:
        print(f"  ⚠ Values outside expected range!\n")

# Summary
print(f"{'='*60}")
print(f"Safety Check Summary")
print(f"{'='*60}")

all_checks_passed = (
    actual_count == expected_count and
    len(unique_dims) == 1 and
    not has_issues and
    zero_vectors == 0
)

if all_checks_passed:
    print(f"✓ All safety checks passed! Embeddings are valid.")
else:
    print(f"⚠ Some checks failed. Review the results above.")
    
print(f"{'='*60}\n")

Running safety checks on embeddings...

Check 1: Count verification
  - Expected embeddings: 67,043
  - Actual embeddings: 64,011
  ⚠ Missing 3,032 embeddings!

Check 2: Dimension verification
  ✓ All embeddings have dimension: 1024
  ✓ Matches expected dimension: 1024

Check 3: Validity check (NaN/Inf)
  ✓ No NaN or Inf values found

Check 4: Normalization verification
  ✓ All embeddings are properly normalized

Check 5: Zero vector check
  ✓ No zero vectors found

Check 6: Value range check
  ✓ All embeddings are properly normalized

Check 5: Zero vector check
  ✓ No zero vectors found

Check 6: Value range check
  - Min value: -0.218597
  - Max value: 0.196340
  - Mean value: -0.000057
  - Std deviation: 0.031250
  ✓ Values within expected range for normalized embeddings [-1, 1]

Safety Check Summary
⚠ Some checks failed. Review the results above.

  - Min value: -0.218597
  - Max value: 0.196340
  - Mean value: -0.000057
  - Std deviation: 0.031250
  ✓ Values within expected range 

## 7b. Fill Missing Embeddings (If Any)

In [ ]:
# Identify and compute embeddings for missing papers
print("Checking for missing embeddings...\n")

# Calculate which papers are missing
expected_count = len(full_texts)
actual_count = sum(emb.shape[0] for emb in embeddings_list)

if actual_count < expected_count:
    missing_count = expected_count - actual_count
    print(f"Found {missing_count:,} missing embeddings")
    print(f"Missing papers are at indices: {actual_count} to {expected_count - 1}\n")
    
    # Identify the missing range
    missing_start = actual_count
    missing_end = expected_count
    missing_texts = full_texts[missing_start:missing_end]
    
    print(f"Processing {len(missing_texts):,} missing papers...")
    print(f"{'='*60}\n")
    
    # Process missing papers in batches
    missing_embeddings = []
    missing_failed_batches = []
    
    start_time = time.time()
    
    try:
        with tqdm(total=len(missing_texts), desc="Computing missing embeddings", unit="paper") as pbar:
            for i in range(0, len(missing_texts), BATCH_SIZE):
                batch_texts = missing_texts[i:i+BATCH_SIZE]
                actual_idx = missing_start + i
                
                try:
                    # Generate embeddings for batch
                    batch_embeddings = embed_batch(
                        texts=batch_texts,
                        instruction=INSTRUCTION,
                        normalize=NORMALIZE
                    )
                    
                    missing_embeddings.append(batch_embeddings)
                    
                    # Update progress bar
                    pbar.update(len(batch_texts))
                    
                except Exception as e:
                    print(f"\n✗ Failed to process batch at index {actual_idx}: {e}")
                    missing_failed_batches.append((actual_idx, actual_idx + len(batch_texts)))
                    pbar.update(len(batch_texts))
                    
    except KeyboardInterrupt:
        print("\n\n⚠ Process interrupted by user")
        print("Partial progress on missing embeddings saved in memory")
        raise
    
    elapsed_time = time.time() - start_time
    
    if missing_embeddings:
        # Add the missing embeddings to the main list
        embeddings_list.extend(missing_embeddings)
        
        print(f"\n✓ Completed processing missing embeddings")
        print(f"  - Time: {elapsed_time:.2f}s")
        print(f"  - Processed: {sum(e.shape[0] for e in missing_embeddings):,} embeddings")
        
        if missing_failed_batches:
            print(f"  - Failed batches: {len(missing_failed_batches)}")
            print(f"  - Failed ranges: {missing_failed_batches}")
        
        # Save a checkpoint with the complete data
        print(f"\n  Saving checkpoint with complete data...")
        save_checkpoint(embeddings_list, expected_count)
        
        # Re-verify count
        new_count = sum(emb.shape[0] for emb in embeddings_list)
        print(f"\n  New total embeddings: {new_count:,}")
        if new_count == expected_count:
            print(f"  ✓ All missing embeddings filled successfully!\n")
        else:
            still_missing = expected_count - new_count
            print(f"  ⚠ Still missing {still_missing:,} embeddings\n")
    else:
        print(f"\n✗ No embeddings were computed for missing papers")
        print(f"  Check the error messages above\n")
        
elif actual_count > expected_count:
    print(f"⚠ Warning: More embeddings ({actual_count:,}) than papers ({expected_count:,})")
    print(f"  Extra embeddings will be truncated when combining.\n")
    
else:
    print(f"✓ No missing embeddings - all {expected_count:,} papers have embeddings!\n")

## 8. Combine and Save Embeddings

In [ ]:
# Combine all embeddings into a single array
print("\nCombining embeddings...")
all_embeddings = np.vstack(embeddings_list)

print(f"✓ Final embeddings shape: {all_embeddings.shape}")
print(f"  - Number of papers: {all_embeddings.shape[0]:,}")
print(f"  - Embedding dimension: {all_embeddings.shape[1]}")
print(f"  - Data type: {all_embeddings.dtype}")
print(f"  - Memory size: {all_embeddings.nbytes / (1024**2):.2f} MB")

# Verify embeddings
print(f"\nEmbedding statistics:")
print(f"  - Min value: {all_embeddings.min():.6f}")
print(f"  - Max value: {all_embeddings.max():.6f}")
print(f"  - Mean value: {all_embeddings.mean():.6f}")
print(f"  - Std deviation: {all_embeddings.std():.6f}")

if NORMALIZE:
    # Check normalization (should be close to 1.0)
    norms = np.linalg.norm(all_embeddings, axis=1)
    print(f"\nNormalization check (should be ~1.0):")
    print(f"  - Mean L2 norm: {norms.mean():.6f}")
    print(f"  - Min L2 norm: {norms.min():.6f}")
    print(f"  - Max L2 norm: {norms.max():.6f}")

In [ ]:
# Save embeddings to file
print(f"\nSaving embeddings to {OUTPUT_EMBEDDINGS}...")
np.save(OUTPUT_EMBEDDINGS, all_embeddings)
file_size = os.path.getsize(OUTPUT_EMBEDDINGS) / (1024**2)
print(f"✓ Embeddings saved successfully ({file_size:.2f} MB)")

# Create metadata
metadata = {
    "created_at": datetime.now().isoformat(),
    "model": "Qwen/Qwen3-Embedding-0.6B",
    "api_url": QWEN_API_URL,
    "embedding_dimension": int(all_embeddings.shape[1]),
    "num_papers": int(all_embeddings.shape[0]),
    "input_file": INPUT_FILE,
    "output_file": OUTPUT_EMBEDDINGS,
    "instruction": INSTRUCTION,
    "normalized": NORMALIZE,
    "batch_size": BATCH_SIZE,
    "processing_time_seconds": elapsed_time,
    "processing_rate_papers_per_second": papers_processed / elapsed_time,
    "failed_batches": failed_batches,
    "dtype": str(all_embeddings.dtype),
    "file_size_mb": file_size,
    "statistics": {
        "min": float(all_embeddings.min()),
        "max": float(all_embeddings.max()),
        "mean": float(all_embeddings.mean()),
        "std": float(all_embeddings.std())
    }
}

# Save metadata
print(f"Saving metadata to {OUTPUT_METADATA}...")
with open(OUTPUT_METADATA, 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2)

print(f"✓ Metadata saved successfully")

print(f"\n{'='*60}")
print(f"✓ ALL DONE!")
print(f"{'='*60}")
print(f"\nOutput files:")
print(f"  1. {OUTPUT_EMBEDDINGS} - NumPy array of embeddings")
print(f"  2. {OUTPUT_METADATA} - Metadata JSON file")
print(f"\nTo load embeddings later:")
print(f"  embeddings = np.load('{OUTPUT_EMBEDDINGS}')")

## 9. Quick Verification & Example Usage

In [ ]:
# Load the saved embeddings to verify
print("Verifying saved embeddings...")
loaded_embeddings = np.load(OUTPUT_EMBEDDINGS)

print(f"✓ Successfully loaded embeddings")
print(f"  - Shape: {loaded_embeddings.shape}")
print(f"  - Data type: {loaded_embeddings.dtype}")

# Verify they match
if np.allclose(loaded_embeddings, all_embeddings):
    print(f"✓ Loaded embeddings match the generated embeddings!")
else:
    print(f"⚠ Warning: Loaded embeddings differ from generated embeddings")

In [ ]:
# Example: Compute similarity between two random papers
print("\nExample: Computing similarity between papers...\n")

# Select two random papers
idx1, idx2 = 0, 100  # You can change these indices

# Get their embeddings
emb1 = loaded_embeddings[idx1]
emb2 = loaded_embeddings[idx2]

# Compute cosine similarity (dot product since they're normalized)
similarity = np.dot(emb1, emb2)

print(f"Paper {idx1}:")
if 'title' in dataset[idx1]:
    print(f"  Title: {dataset[idx1]['title'][:100]}...")
print(f"  Text preview: {full_texts[idx1][:150]}...\n")

print(f"Paper {idx2}:")
if 'title' in dataset[idx2]:
    print(f"  Title: {dataset[idx2]['title'][:100]}...")
print(f"  Text preview: {full_texts[idx2][:150]}...\n")

print(f"Cosine similarity: {similarity:.4f}")
print(f"(Range: -1 to 1, where 1 = identical, 0 = orthogonal, -1 = opposite)")

In [ ]:
# Example: Find most similar papers to a given paper
print("\nExample: Finding most similar papers...\n")

query_idx = 0  # Change this to any paper index
top_k = 5  # Number of similar papers to find

# Get query embedding
query_emb = loaded_embeddings[query_idx]

# Compute similarities with all papers
similarities = np.dot(loaded_embeddings, query_emb)

# Get top-k most similar (excluding the query itself)
top_indices = np.argsort(similarities)[::-1][1:top_k+1]  # Skip first (itself)

print(f"Query paper (index {query_idx}):")
if 'title' in dataset[query_idx]:
    print(f"  Title: {dataset[query_idx]['title']}")
print(f"  Text preview: {full_texts[query_idx][:200]}...\n")

print(f"Top {top_k} most similar papers:\n")
for rank, idx in enumerate(top_indices, 1):
    print(f"{rank}. Paper {idx} (similarity: {similarities[idx]:.4f})")
    if 'title' in dataset[idx]:
        print(f"   Title: {dataset[idx]['title']}")
    print(f"   Text preview: {full_texts[idx][:150]}...")
    print()

## 10. Clean Up Checkpoints (Optional)

In [ ]:
# Optionally remove checkpoint files after successful completion
REMOVE_CHECKPOINTS = False  # Set to True to remove checkpoints

if REMOVE_CHECKPOINTS:
    checkpoint_files = list(Path(CHECKPOINT_DIR).glob("checkpoint_*.npy"))
    if checkpoint_files:
        print(f"Removing {len(checkpoint_files)} checkpoint files...")
        for checkpoint_file in checkpoint_files:
            checkpoint_file.unlink()
            print(f"  Removed: {checkpoint_file.name}")
        print(f"✓ Checkpoints cleaned up")
    else:
        print("No checkpoint files to remove")
else:
    print("Keeping checkpoint files (set REMOVE_CHECKPOINTS=True to remove)")
    checkpoint_files = list(Path(CHECKPOINT_DIR).glob("checkpoint_*.npy"))
    if checkpoint_files:
        print(f"  Found {len(checkpoint_files)} checkpoint files in {CHECKPOINT_DIR}/")